# Facial Recognition Classification — MLOps Summative
**African Leadership University — BSE**

**Team:** Edwin, Glory, Justine, Mike

This notebook covers the full offline ML pipeline:
1. Environment Setup
2. Data Acquisition & Loading
3. **Offline Data Augmentation** (expanding a small dataset)
4. Exploratory Data Analysis (EDA) with Feature Interpretations
5. Data Preprocessing
6. Model Creation (MobileNetV2 Transfer Learning)
7. Model Training with Optimization
8. Comprehensive Evaluation (7 metrics)
9. Model Export for Deployment
10. Prediction & Retraining Functions

## 1. Environment Setup

In [ ]:
!pip install -q tensorflow matplotlib seaborn scikit-learn pillow

import os, json, shutil, warnings, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, array_to_img
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, accuracy_score, roc_curve, auc)
from sklearn.preprocessing import label_binarize
from PIL import Image, ImageEnhance, ImageFilter

warnings.filterwarnings('ignore')
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## 2. Data Acquisition

Our dataset consists of face images of 4 team members. Each member provided 1-3 original images.

Since we have a **very small original dataset**, we will use **offline data augmentation** in the next section to expand it to ~35-50 images per class before training.

**Instructions:**
1. Mount Google Drive
2. Set `ORIGINAL_DIR` to where your original images are
3. The augmentation step will create a new expanded dataset in `DATASET_DIR`

In [ ]:
# --- Mount Google Drive (uncomment on Colab) ---
from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURATION ===
# Path to your ORIGINAL images (1-3 per person)
# Update this to match your Drive structure
ORIGINAL_DIR = '/content/drive/MyDrive/train_images'
# If your images are in Shared with me, add shortcut to My Drive first
# Then use: '/content/drive/MyDrive/your_shortcut_name'

# Path where the AUGMENTED dataset will be saved
DATASET_DIR = '/content/augmented_dataset'

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 30
SEED = 42

# Check what we have
print("Original images found:")
total_originals = 0
for cls in sorted(os.listdir(ORIGINAL_DIR)):
    cls_path = os.path.join(ORIGINAL_DIR, cls)
    if os.path.isdir(cls_path):
        imgs = [f for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.webp','.heic'))]
        print(f"  {cls}: {len(imgs)} images")
        total_originals += len(imgs)
print(f"\nTotal originals: {total_originals}")
print("\n⚠️  This is a very small dataset — we'll augment it in the next step.")

## 3. Offline Data Augmentation

With only 1-3 images per person, we need to **expand our dataset** before training. This is a standard and widely-used technique in computer vision when data is limited.

**Augmentation techniques applied:**
- Random rotation (±30°)
- Horizontal flip
- Brightness adjustment (±30%)
- Contrast adjustment (±30%)
- Color/saturation jitter
- Gaussian blur
- Random zoom/crop
- Width/height shift

**Target:** ~35 augmented images per original image → ~40-70 total per class.

This is NOT cheating — it's a core ML preprocessing technique. The model still learns from real facial features; augmentation just teaches it to be **invariant** to lighting, angle, and scale.

In [ ]:
# === AUGMENTATION ENGINE ===

def augment_image(img, num_augmented=35):
    """Generate multiple augmented versions of a single PIL image."""
    augmented = []
    width, height = img.size

    for i in range(num_augmented):
        aug = img.copy()

        # --- Random rotation (±30°) ---
        if random.random() > 0.3:
            angle = random.uniform(-30, 30)
            aug = aug.rotate(angle, fillcolor=(128, 128, 128))

        # --- Horizontal flip (50% chance) ---
        if random.random() > 0.5:
            aug = aug.transpose(Image.FLIP_LEFT_RIGHT)

        # --- Brightness (±30%) ---
        if random.random() > 0.3:
            factor = random.uniform(0.7, 1.3)
            aug = ImageEnhance.Brightness(aug).enhance(factor)

        # --- Contrast (±30%) ---
        if random.random() > 0.4:
            factor = random.uniform(0.7, 1.3)
            aug = ImageEnhance.Contrast(aug).enhance(factor)

        # --- Color/Saturation jitter ---
        if random.random() > 0.5:
            factor = random.uniform(0.8, 1.2)
            aug = ImageEnhance.Color(aug).enhance(factor)

        # --- Gaussian blur (light) ---
        if random.random() > 0.7:
            radius = random.uniform(0.5, 1.5)
            aug = aug.filter(ImageFilter.GaussianBlur(radius=radius))

        # --- Random crop & resize (simulates zoom) ---
        if random.random() > 0.4:
            crop_frac = random.uniform(0.8, 0.95)
            new_w = int(width * crop_frac)
            new_h = int(height * crop_frac)
            left = random.randint(0, width - new_w)
            top = random.randint(0, height - new_h)
            aug = aug.crop((left, top, left + new_w, top + new_h))
            aug = aug.resize((width, height), Image.LANCZOS)

        # --- Sharpness ---
        if random.random() > 0.6:
            factor = random.uniform(0.8, 1.5)
            aug = ImageEnhance.Sharpness(aug).enhance(factor)

        augmented.append(aug)

    return augmented


def expand_dataset(original_dir, output_dir, augment_per_image=35, target_size=(224, 224)):
    """
    Read original images, generate augmented versions, save to output_dir.
    """
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    classes = sorted([d for d in os.listdir(original_dir)
                      if os.path.isdir(os.path.join(original_dir, d))])

    print(f"Augmenting {len(classes)} classes...")
    print(f"Target: ~{augment_per_image} augmented images per original\n")

    stats = {}
    for cls in classes:
        cls_in = os.path.join(original_dir, cls)
        cls_out = os.path.join(output_dir, cls)
        os.makedirs(cls_out, exist_ok=True)

        img_files = [f for f in os.listdir(cls_in)
                     if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.webp'))]

        count = 0
        for img_file in img_files:
            img_path = os.path.join(cls_in, img_file)
            try:
                img = Image.open(img_path).convert('RGB')
                img = img.resize(target_size, Image.LANCZOS)
            except Exception as e:
                print(f"  ⚠️ Skipping {img_file}: {e}")
                continue

            # Save the original (resized)
            base_name = os.path.splitext(img_file)[0]
            img.save(os.path.join(cls_out, f"{base_name}_original.jpg"), "JPEG", quality=95)
            count += 1

            # Generate augmented versions
            augmented = augment_image(img, num_augmented=augment_per_image)
            for j, aug_img in enumerate(augmented):
                aug_img.save(os.path.join(cls_out, f"{base_name}_aug_{j:03d}.jpg"),
                            "JPEG", quality=90)
                count += 1

        stats[cls] = count
        print(f"  ✅ {cls}: {len(img_files)} originals → {count} total images")

    print(f"\n📊 Dataset expanded: {sum(stats.values())} total images")
    return stats

In [ ]:
# === RUN AUGMENTATION ===
augment_stats = expand_dataset(
    original_dir=ORIGINAL_DIR,
    output_dir=DATASET_DIR,
    augment_per_image=35,   # 35 augmented per original image
    target_size=(IMG_SIZE, IMG_SIZE)
)

# Update class info
CLASS_NAMES = sorted([d for d in os.listdir(DATASET_DIR)
                      if os.path.isdir(os.path.join(DATASET_DIR, d))])
NUM_CLASSES = len(CLASS_NAMES)
print(f"\nClasses: {CLASS_NAMES}")
print(f"Num classes: {NUM_CLASSES}")

In [ ]:
# --- Visualize: Original vs Augmented samples ---
fig, axes = plt.subplots(NUM_CLASSES, 6, figsize=(18, 3.5*NUM_CLASSES))
fig.suptitle('Original + Augmented Samples per Class', fontsize=16, fontweight='bold', y=1.02)

for i, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(DATASET_DIR, cls)
    all_imgs = sorted(os.listdir(cls_path))

    # First col = original, rest = augmented
    originals = [f for f in all_imgs if '_original' in f]
    augmented = [f for f in all_imgs if '_aug_' in f]

    show_files = originals[:1] + augmented[:5]
    labels = ['Original'] + [f'Aug {j+1}' for j in range(5)]

    for j in range(6):
        ax = axes[i][j] if NUM_CLASSES > 1 else axes[j]
        if j < len(show_files):
            img = load_img(os.path.join(cls_path, show_files[j]), target_size=(IMG_SIZE, IMG_SIZE))
            ax.imshow(img)
            ax.set_title(labels[j], fontsize=9)
        ax.axis('off')
        if j == 0:
            ax.set_ylabel(cls, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('augmentation_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Exploratory Data Analysis (EDA)

Now analyzing our **augmented** dataset.

### Feature Interpretations:
1. **Class Distribution** — Shows whether our augmentation produced a balanced dataset across all team members. Balance is critical to prevent the model from being biased toward any one person.
2. **Image Dimension & Aspect Ratio** — All images are now standardized to 224×224, confirming our preprocessing pipeline works correctly.
3. **Mean Pixel Intensity per Class** — Reveals if certain team members' photos have consistently brighter or darker lighting. Large differences here indicate the model might learn lighting shortcuts instead of facial features — our augmentation addresses this with brightness jittering.

In [ ]:
# Count images per class (augmented dataset)
class_counts = {}
for cls in CLASS_NAMES:
    cls_path = os.path.join(DATASET_DIR, cls)
    count = len([f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])
    class_counts[cls] = count
print("Augmented dataset counts:")
for cls, cnt in class_counts.items():
    print(f"  {cls}: {cnt} images")
print(f"Total: {sum(class_counts.values())} images")

In [ ]:
# --- Class Distribution (Bar + Pie) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63'][:NUM_CLASSES]

bars = axes[0].bar(class_counts.keys(), class_counts.values(), color=colors, edgecolor='black')
axes[0].set_title('Image Count per Team Member (After Augmentation)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for bar, c in zip(bars, class_counts.values()):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                str(c), ha='center', fontweight='bold')

axes[1].pie(class_counts.values(), labels=class_counts.keys(), autopct='%1.1f%%',
            colors=colors, startangle=90, explode=[0.05]*NUM_CLASSES)
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Image Properties + Mean Brightness per Class ---
class_brightness = {cls: [] for cls in CLASS_NAMES}

for cls in CLASS_NAMES:
    cls_path = os.path.join(DATASET_DIR, cls)
    for f in os.listdir(cls_path):
        if f.lower().endswith(('.jpg','.jpeg','.png','.bmp')):
            try:
                arr = img_to_array(load_img(os.path.join(cls_path, f),
                                             target_size=(IMG_SIZE, IMG_SIZE)))
                class_brightness[cls].append(arr.mean())
            except: pass

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Brightness per class
bright_means = [np.mean(class_brightness[c]) for c in CLASS_NAMES]
bright_stds = [np.std(class_brightness[c]) for c in CLASS_NAMES]
axes[0].bar(CLASS_NAMES, bright_means, yerr=bright_stds, color=colors,
            edgecolor='black', capsize=5)
axes[0].set_title('Mean Pixel Intensity by Class', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Mean Pixel Value (0-255)')

# Brightness distributions as violin/box
brightness_data = [class_brightness[c] for c in CLASS_NAMES]
bp = axes[1].boxplot(brightness_data, labels=CLASS_NAMES, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Brightness Distribution per Class', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Mean Pixel Value')

plt.tight_layout()
plt.savefig('image_properties.png', dpi=150, bbox_inches='tight')
plt.show()

for cls in CLASS_NAMES:
    print(f"  {cls}: mean brightness = {np.mean(class_brightness[cls]):.1f} ± {np.std(class_brightness[cls]):.1f}")

In [ ]:
# --- Sample Images Grid ---
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(15, 3*NUM_CLASSES))
fig.suptitle('Sample Images per Team Member', fontsize=16, fontweight='bold', y=1.02)

for i, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(DATASET_DIR, cls)
    imgs = sorted([f for f in os.listdir(cls_path)
                   if f.lower().endswith(('.jpg','.jpeg','.png'))])
    # Pick a mix: 1 original + 4 random augmented
    originals = [f for f in imgs if '_original' in f][:1]
    aug = random.sample([f for f in imgs if '_aug_' in f], min(4, len(imgs)-1))
    show = originals + aug

    for j in range(5):
        ax = axes[i][j] if NUM_CLASSES > 1 else axes[j]
        if j < len(show):
            img = load_img(os.path.join(cls_path, show[j]), target_size=(IMG_SIZE, IMG_SIZE))
            ax.imshow(img)
        ax.axis('off')
        if j == 0:
            ax.set_ylabel(cls, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Data Preprocessing

**Preprocessing pipeline:**
- All images already resized to 224×224 during augmentation
- Normalize pixel values to [0, 1]
- 80/20 train/validation split
- **Additional real-time augmentation** during training (on top of our offline augmentation) for maximum variety

In [ ]:
# --- Data Generators ---
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.9, 1.1],
    fill_mode='nearest',
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATASET_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True, seed=SEED
)

val_generator = val_datagen.flow_from_directory(
    DATASET_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=False, seed=SEED
)

print(f"Training samples:   {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Class indices: {train_generator.class_indices}")

## 6. Model Creation — Transfer Learning (MobileNetV2)

**Architecture:** MobileNetV2 pre-trained on ImageNet + custom classification head.

**Optimization techniques used:**

| Technique | Detail |
|---|---|
| Transfer Learning | MobileNetV2 (ImageNet weights) |
| Fine-Tuning | Unfreeze layers 100+ |
| L2 Regularization | λ = 0.01 on Dense layers |
| Dropout | 50% + 30% |
| Batch Normalization | After pooling & between Dense layers |
| Early Stopping | patience=7, restore_best_weights |
| LR Scheduling | ReduceLROnPlateau (0.5×, patience=3) |
| Data Augmentation | 8 offline + 6 real-time techniques |

In [ ]:
def build_model(num_classes, fine_tune_at=100):
    """Build MobileNetV2-based facial recognition classifier."""
    base_model = MobileNetV2(weights='imagenet', include_top=False,
                             input_shape=(IMG_SIZE, IMG_SIZE, 3))

    # Freeze early layers, fine-tune deeper layers
    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False
    for layer in base_model.layers[fine_tune_at:]:
        layer.trainable = True

    # Custom classification head with regularization
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = Dropout(0.5)(x)
    x = BatchNormalization()(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)

    return Model(inputs=base_model.input, outputs=output)

model = build_model(NUM_CLASSES)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

trainable = sum(p.numpy().size for p in model.trainable_weights)
total = model.count_params()
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,}")
print(f"Frozen params:    {total - trainable:,}")

## 7. Model Training

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print(f"\nCompleted at epoch {len(history.history['loss'])}")
print(f"Best val_accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Best val_loss:     {min(history.history['val_loss']):.4f}")

In [ ]:
# --- Training Curves ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_pairs = [
    ('accuracy','val_accuracy','Accuracy'),
    ('loss','val_loss','Loss'),
    ('precision','val_precision','Precision'),
    ('recall','val_recall','Recall')
]
for ax, (tr, va, title) in zip(axes.flat, metrics_pairs):
    ax.plot(history.history[tr], label=f'Train {title}', linewidth=2)
    ax.plot(history.history[va], label=f'Val {title}', linewidth=2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Comprehensive Model Evaluation

**Metrics computed:**
1. Accuracy
2. Loss (categorical cross-entropy)
3. Precision (per-class + weighted)
4. Recall (per-class + weighted)
5. F1-Score (per-class + weighted)
6. Confusion Matrix
7. ROC-AUC (per-class, one-vs-rest)

In [ ]:
# Load best model
model = load_model('best_model.keras')

# Predictions
val_generator.reset()
y_pred_probs = model.predict(val_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_generator.classes

val_loss, val_acc, val_prec, val_rec = model.evaluate(val_generator, verbose=0)
val_f1 = f1_score(y_true, y_pred, average='weighted')

print("=" * 55)
print("        EVALUATION RESULTS — Validation Set")
print("=" * 55)
print(f"  Accuracy:    {val_acc:.4f}  ({val_acc*100:.2f}%)")
print(f"  Loss:        {val_loss:.4f}")
print(f"  Precision:   {val_prec:.4f}")
print(f"  Recall:      {val_rec:.4f}")
print(f"  F1-Score:    {val_f1:.4f}")
print("=" * 55)

In [ ]:
# --- Classification Report ---
print("\nDetailed Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0],
            linewidths=1, linecolor='white')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1],
            linewidths=1, linecolor='white')
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.suptitle('Confusion Matrix Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Per-Class Metrics Bar Chart ---
report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
x = np.arange(NUM_CLASSES)
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x-width, [report_dict[c]['precision'] for c in CLASS_NAMES], width,
       label='Precision', color='#2196F3', edgecolor='black')
ax.bar(x, [report_dict[c]['recall'] for c in CLASS_NAMES], width,
       label='Recall', color='#4CAF50', edgecolor='black')
ax.bar(x+width, [report_dict[c]['f1-score'] for c in CLASS_NAMES], width,
       label='F1-Score', color='#FF9800', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.15)
ax.set_title('Per-Class Evaluation Metrics', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- ROC Curve (One-vs-Rest) ---
y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))

fig, ax = plt.subplots(figsize=(8, 6))
for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, linewidth=2, label=f'{cls} (AUC={roc_auc:.3f})')

ax.plot([0,1], [0,1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (One-vs-Rest)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Model Export for Deployment

In [ ]:
# Save model
model.save('facial_recognition_model.keras')
print("Model saved: facial_recognition_model.keras")

# Save class mapping
class_mapping = {str(v): k for k, v in train_generator.class_indices.items()}
with open('class_names.json', 'w') as f:
    json.dump(class_mapping, f, indent=2)
print(f"Class mapping: {class_mapping}")

# Save training config
config = {
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'class_indices': train_generator.class_indices,
    'best_val_accuracy': float(max(history.history['val_accuracy'])),
    'best_val_loss': float(min(history.history['val_loss'])),
    'epochs_trained': len(history.history['loss']),
    'augmentation': 'offline (35 per image) + real-time',
    'original_images_per_class': {cls: len([f for f in os.listdir(os.path.join(ORIGINAL_DIR, cls))
                                             if f.lower().endswith(('.jpg','.jpeg','.png'))])
                                   for cls in CLASS_NAMES if os.path.exists(os.path.join(ORIGINAL_DIR, cls))}
}
with open('training_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print("Config saved: training_config.json")

print("\n✅ All artifacts exported. Ready for deployment!")

## 10. Download Model Files
Run this cell to download the 3 files you need for deployment.

In [ ]:
from google.colab import files

# Download model + config files
files.download('facial_recognition_model.keras')
files.download('class_names.json')
files.download('training_config.json')

print("\n📥 Download the 3 files above and place them in your project's models/ folder")

## 11. Prediction & Retraining Functions
These are mirrored in `src/prediction.py` and `src/model.py` for the deployed app.

In [ ]:
def predict_face(image_path, model_path='facial_recognition_model.keras',
                 class_map_path='class_names.json'):
    model = load_model(model_path)
    with open(class_map_path) as f:
        class_names = json.load(f)

    img = load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)[0]
    idx = int(np.argmax(preds))
    return {
        'label': class_names[str(idx)],
        'confidence': float(preds[idx]),
        'all_probabilities': {class_names[str(i)]: float(preds[i]) for i in range(len(preds))}
    }

# Test: predict on a validation image
val_dir = os.path.join(DATASET_DIR, CLASS_NAMES[0])
test_img = os.path.join(val_dir, os.listdir(val_dir)[0])
result = predict_face(test_img)
print(f"Test prediction: {result['label']} ({result['confidence']*100:.1f}%)")

In [ ]:
def retrain_model(dataset_dir, model_path='facial_recognition_model.keras',
                  epochs=15, learning_rate=0.00005):
    train_dg = ImageDataGenerator(
        rescale=1./255, rotation_range=20, width_shift_range=0.2,
        height_shift_range=0.2, zoom_range=0.2, horizontal_flip=True,
        brightness_range=[0.8,1.2], validation_split=0.2)
    val_dg = ImageDataGenerator(rescale=1./255, validation_split=0.2)

    train_gen = train_dg.flow_from_directory(
        dataset_dir, target_size=(IMG_SIZE,IMG_SIZE),
        batch_size=16, class_mode='categorical', subset='training', seed=42)
    val_gen = val_dg.flow_from_directory(
        dataset_dir, target_size=(IMG_SIZE,IMG_SIZE),
        batch_size=16, class_mode='categorical', subset='validation', seed=42)

    model = load_model(model_path)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
                           tf.keras.metrics.Recall(name='recall')])

    history = model.fit(train_gen, epochs=epochs, validation_data=val_gen,
                        callbacks=[EarlyStopping(monitor='val_loss', patience=5,
                                                 restore_best_weights=True)], verbose=1)
    model.save(model_path)
    mapping = {str(v): k for k, v in train_gen.class_indices.items()}
    with open('class_names.json', 'w') as f:
        json.dump(mapping, f, indent=2)

    return {
        'accuracy': float(max(history.history['val_accuracy'])),
        'loss': float(min(history.history['val_loss'])),
        'epochs': len(history.history['loss'])
    }

print("Retraining function defined. Ready for deployment use.")

## Summary

| Technique | Details |
|---|---|
| **Offline Augmentation** | 8 techniques, ~35 images generated per original |
| Transfer Learning | MobileNetV2 (ImageNet) |
| Fine-Tuning | Layers 100+ unfrozen |
| L2 Regularization | λ=0.01 |
| Dropout | 50% + 30% |
| Batch Normalization | 2 layers |
| Early Stopping | patience=7 |
| LR Scheduling | ReduceLROnPlateau |
| Real-time Augmentation | 6 additional techniques during training |

**Evaluation Metrics:** Accuracy, Loss, Precision, Recall, F1-Score, Confusion Matrix, ROC-AUC

**Original dataset:** ~1-3 images per class → **Augmented to ~36-108 per class**